# Train Second Read's own AI detector

This notebook trains **our own** AI-writing detector from **our own database** (Supabase).

**How to run it:** open it in [Google Colab](https://colab.research.google.com) (File → Upload notebook), then
**Runtime → Change runtime type → T4 GPU** (free), and run the cells from top to bottom.

1. Connect to the database and Hugging Face
2. *(first time only)* fill the database with free public essays
3. Pull the labelled texts out of the database
4. Train the model (about 15–25 minutes on the free GPU)
5. Check how often it wrongly flags human writing
6. Save the model to your Hugging Face account so the site can use it

Run it again whenever the database has grown (e.g. once a month) to make the detector better.

In [ ]:
!pip -q install -U transformers datasets accelerate huggingface_hub scikit-learn

## 1. Connect
Paste your Supabase **Project URL** and **secret key** (Supabase → Project Settings → API Keys), and log in to Hugging Face with a **write** token (huggingface.co → Settings → Access Tokens). Nothing you type here is saved in the notebook.

In [ ]:
import getpass, hashlib, re, json, random, requests
SUPABASE_URL = input("Supabase project URL: ").strip().rstrip("/")
SUPABASE_KEY = getpass.getpass("Supabase secret key: ").strip()
HF_REPO = input("Name for your model on Hugging Face (e.g. your-name/second-read-detector): ").strip()

from huggingface_hub import login
login()  # paste a Hugging Face token with WRITE access

H = {"apikey": SUPABASE_KEY, "content-type": "application/json"}
if SUPABASE_KEY.startswith("eyJ"):
    H["authorization"] = "Bearer " + SUPABASE_KEY

def norm(s):  # must match api/_db.js so duplicates are detected
    s = s.lower().replace("\u2018", "'").replace("\u2019", "'")
    s = re.sub(r"[^a-z0-9' ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def sha(s): return hashlib.sha256(s.encode()).hexdigest()
def words(s): return len(re.findall(r"[A-Za-z0-9'-]+", s))

r = requests.get(f"{SUPABASE_URL}/rest/v1/training_stats?select=*", headers=H)
print("Connected!" if r.ok else f"Problem: {r.status_code} {r.text}")
print(r.json() if r.ok else "")

## 2. (First time only) Fill the database with public essays
Adds human-written and AI-written essays from the free public dataset `artem9k/ai-text-detection-pile`. Duplicates are skipped, so running it twice is safe. Increase `PER_LABEL` for a stronger model.

In [ ]:
from datasets import load_dataset
PER_LABEL = 8000
DATASET = "artem9k/ai-text-detection-pile"

ds = load_dataset(DATASET, split="train", streaming=True).shuffle(seed=42, buffer_size=20000)
counts, batch = {"human": 0, "ai": 0}, []
def flush():
    global batch
    if not batch: return
    r = requests.post(f"{SUPABASE_URL}/rest/v1/samples?on_conflict=text_hash",
                      headers={**H, "prefer": "resolution=ignore-duplicates,return=minimal"}, data=json.dumps(batch))
    if not r.ok: print("insert problem:", r.status_code, r.text[:200])
    batch = []
for row in ds:
    label = "human" if str(row["source"]).lower() == "human" else "ai"
    text = (row["text"] or "").strip()
    if counts[label] >= PER_LABEL or not (60 <= words(text) <= 1500): continue
    counts[label] += 1
    batch.append({"text": text, "label": label, "origin": "public:" + DATASET, "text_hash": sha(norm(text)), "words": words(text)})
    if len(batch) >= 400: flush(); print(counts)
    if all(v >= PER_LABEL for v in counts.values()): break
flush(); print("done", counts)

## 3. Pull the labelled training data from our database

In [ ]:
rows, page = [], 1000
while True:
    r = requests.get(f"{SUPABASE_URL}/rest/v1/samples?select=id,text,label,origin&label=not.is.null&order=id",
                     headers={**H, "range-unit": "items", "range": f"{len(rows)}-{len(rows) + page - 1}"})
    r.raise_for_status(); got = r.json(); rows += got
    if len(got) < page: break
from collections import Counter
print(len(rows), "labelled texts:", Counter(r["label"] for r in rows))
print(Counter(r["origin"] for r in rows))

### Cut texts into sentence-sized pieces
The site scores **one sentence at a time**, so we train on pieces of 1–3 sentences plus some whole paragraphs. Pieces from the same text stay on the same side of the train/test split, so the test is fair.

In [ ]:
def pieces(text, rng):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", text)) if words(s) >= 5]
    out, i = [], 0
    while i < len(sents):
        n = rng.choice([1, 1, 2, 3]); chunk = " ".join(sents[i:i + n]); i += n
        if 6 <= words(chunk) <= 120: out.append(chunk)
    paras = [p for p in text.split("\n") if 40 <= words(p) <= 200]
    return out[:12] + paras[:2]

rng = random.Random(0)
ids = sorted({r["id"] for r in rows}); rng.shuffle(ids)
test_ids = set(ids[: max(1, len(ids) // 10)])
train, test = [], []
for r in rows:
    y = 1 if r["label"] == "ai" else 0
    for p in pieces(r["text"], rng):
        (test if r["id"] in test_ids else train).append({"text": p, "label": y})
def balance(xs):
    h = [x for x in xs if x["label"] == 0]; a = [x for x in xs if x["label"] == 1]; n = min(len(h), len(a))
    rng.shuffle(h); rng.shuffle(a); out = h[:n] + a[:n]; rng.shuffle(out); return out
train, test = balance(train)[:120000], balance(test)[:8000]
print(len(train), "training pieces,", len(test), "test pieces")

## 4. Train

In [ ]:
import numpy as np, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

BASE = "distilroberta-base"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2, id2label={0: "human", 1: "ai"}, label2id={"human": 0, "ai": 1})
enc = lambda b: tok(b["text"], truncation=True, max_length=256)
dtrain = Dataset.from_list(train).map(enc, batched=True, remove_columns=["text"])
dtest = Dataset.from_list(test).map(enc, batched=True, remove_columns=["text"])

args = TrainingArguments(output_dir="out", num_train_epochs=2, learning_rate=3e-5, per_device_train_batch_size=32,
                         per_device_eval_batch_size=64, warmup_ratio=0.06, weight_decay=0.01, fp16=torch.cuda.is_available(),
                         eval_strategy="epoch", save_strategy="no", logging_steps=200, report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=dtrain, eval_dataset=dtest, data_collator=DataCollatorWithPadding(tok))
trainer.train()

## 5. Check it
The most important number for students is the **false-alarm rate**: how often real human writing is flagged as AI. We want this low.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
pred = trainer.predict(dtest)
p_ai = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy(); y = np.array(pred.label_ids)
print(f"Accuracy: {accuracy_score(y, p_ai >= 0.5):.1%}   AUC: {roc_auc_score(y, p_ai):.3f}")
for t in (0.5, 0.7, 0.9):
    fa = ((p_ai >= t) & (y == 0)).sum() / max(1, (y == 0).sum()); hit = ((p_ai >= t) & (y == 1)).sum() / max(1, (y == 1).sum())
    print(f"threshold {t}: flags {hit:.1%} of AI text, false alarms on {fa:.1%} of human text")

def check(t):
    with torch.inference_mode():
        b = tok([t], return_tensors="pt", truncation=True, max_length=256).to(model.device)
        return float(torch.softmax(model(**b).logits, -1)[0, 1])
print(check("Honestly, I didn't expect the lab to take three hours, but our titration kept overshooting."))
print(check("In today's rapidly evolving landscape, it is important to note that technology plays a pivotal role."))

## 6. Save the model to Hugging Face
The site's detector Space (see `detector-space/` in the repo) loads it from here.

In [ ]:
model.push_to_hub(HF_REPO, commit_message="Train on Second Read database")
tok.push_to_hub(HF_REPO)
print("Saved: https://huggingface.co/" + HF_REPO)

## 7. (After the Space is running) Test the live detector

In [ ]:
SPACE_URL = input("Your Space URL, e.g. https://your-name-second-read-detector.hf.space: ").strip().rstrip("/")
KEY = getpass.getpass("DETECTOR_KEY (leave empty if you didn't set one): ").strip()
r = requests.post(SPACE_URL + "/score", json={"texts": ["I think the experiment failed because we rushed it.", "Furthermore, it is important to note that this multifaceted issue plays a crucial role."]},
                  headers={"authorization": "Bearer " + KEY} if KEY else {})
print(r.status_code, r.text)